## Feature: Structured Output (`output_pydantic`)

In v1, our Log Analyzer produces a raw text report. It works, but what happens when you need to extract specific fields from that text — like the root cause, or the number of errors? Let's run it and see the problem.

In [1]:
import os

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

In [2]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

---

### v1 approach — raw text output

This is exactly how v1 works. The task produces a raw text report. Let's run it and then try to extract specific fields.

In [3]:
v1_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

v1_crew = Crew(
    agents=[log_analyzer],
    tasks=[v1_task],
    process=Process.sequential,
    verbose=False,
)

v1_result = v1_crew.kickoff(inputs={"log_data": LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Detailed Analysis Report:                                                                                      │
│                                                                                                                 │
│  - **Primary Issue Description:**                                                                               │
│    The primary issue encountered during the deployment process of the application `myapp-deployment` was a      │
│  failure to start due to image pulling issues. This caused the deployment to ultimately fail, requiring a       │
│  rollback.                                                                                                      │
│                                                                                                                 │
│  - **Key Error Messages and Codes:**                                                                            │
│    1. `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`                   │
│    2. `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository     │
│  does not exist or may require 'docker login'`                                                                  │
│    3. `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`          │
│    4. `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its   │
│  progress deadline`                                                                                             │
│    5. `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated`                   │
│                                                                                                                 │
│  - **Timeline of Failure Events:**                                                                              │
│    1. **14:32:15.123** - Deployment of `myapp-deployment` initiated.                                            │
│    2. **14:32:16.567** - Pod `myapp-deployment-7b8c9d5f4-abc12` entered a `Pending` state.                      │
│    3. **14:32:17.890** - Pod failed to start.                                                                   │
│    4. **14:32:18.123** - Image pulling failed due to denied access or non-existent repository.                  │
│    5. **14:32:18.456** - Pod status changed to `ImagePullBackOff` due to image pull failures.                   │
│    6. **14:32:25.901** - Deployment rollout failed as the progress deadline was exceeded.                       │
│    7. **14:32:26.789** - Service `myapp-service` has no available endpoints.                                    │
│    8. **14:32:29.456** - Critical failure triggered rollback.                                                   │
│                                                                                                                 │
│  - **Root Cause Analysis:**                                                                                     │
│    The failure to deploy the application was primarily due to an image pulling failure, as indicated by the     │
│  `ERROR: Failed to pull image "myapp:v1.2.3"`. The error message specifies that the pull access was denied      │
│  because the repository either does not exist or requir

In [4]:
# This is what v1 gives you — a raw text blob
print(v1_result.raw)

Detailed Analysis Report:

- **Primary Issue Description:**
  The primary issue encountered during the deployment process of the application `myapp-deployment` was a failure to start due to image pulling issues. This caused the deployment to ultimately fail, requiring a rollback.

- **Key Error Messages and Codes:**
  1. `[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start`
  2. `[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'`
  3. `[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`
  4. `[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`
  5. `[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated`

- **Timeline of Failure Events:**
  1. **14:32:15.123** - Deployment of `myapp-deployment` initiated.
  2. **14:

In [5]:
# Now try to extract specific fields from that text...

# Want the root cause? Parse the string:
lines = v1_result.raw.split("\n")
for line in lines:
    if "root cause" in line.lower():
        print(f"Found: {line}")
        break

# Want the number of errors? How do you count them in free-form text?
# Want to pass typed data to the next agent? Impossible.
# What if the agent formats it differently next run? Everything breaks.

Found: - **Root Cause Analysis:**


---

### v2 approach — add `output_pydantic`

Same agent. Same log input. But now we define a Pydantic model and tell the task to use it. One parameter changes everything.

In [6]:
from pydantic import BaseModel, Field


class ErrorEntry(BaseModel):
    message: str = Field(description="The error message text")
    severity: str = Field(description="ERROR, CRITICAL, or WARNING")
    timestamp: str = Field(description="When the error occurred")


class LogAnalysisReport(BaseModel):
    primary_issue: str = Field(description="One-line description of the main issue")
    root_cause: str = Field(description="Root cause analysis based on log evidence")
    errors: list[ErrorEntry] = Field(description="All errors found in the log")
    affected_components: list[str] = Field(description="System components affected")
    timeline: list[str] = Field(description="Sequence of events leading to failure")

In [7]:
v2_task = Task(
    description="Analyze the following log data to identify issues:\n{log_data}",
    expected_output="A structured log analysis report",
    output_pydantic=LogAnalysisReport,
    agent=log_analyzer,
)

v2_crew = Crew(
    agents=[log_analyzer],
    tasks=[v2_task],
    process=Process.sequential,
    verbose=False,
)

v2_result = v2_crew.kickoff(inputs={"log_data": LOG_INPUT})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "primary_issue": "Deployment failure due to image pull error and exceeded deployment progress deadline.",    │
│    "root_cause": "Image pull access was denied because the repository 'myapp:v1.2.3' does not exist or          │
│  requires docker login.",                                                                                       │
│    "errors": [                                                                                                  │
│      {                                                                                                          │
│        "message": "Pod myapp-deployment-7b8c9d5f4-abc12 failed to start",                                       │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:17.890"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Failed to pull image \"myapp:v1.2.3\": pull access denied, repository does not exist or may  │
│  require 'docker login'",                                                                                       │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:18.123"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff",                              │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:18.456"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Deployment rollout failed: deployment \"myapp-deployment\" exceeded its progress deadline",  │
│        "severity": "ERROR",                                                                                     │
│        "timestamp": "2024-01-15 14:32:25.901"                                                                   │
│      },                                                                                                         │
│      {                                                                                                          │
│        "message": "Production deployment failed - rollback initiated",                                          │
│        "severity": "CRITICAL",                                                                                  │
│        "timestamp": "2024-01-15 14:32:29.456"          

In [8]:
# Clean, typed access — never breaks
report = v2_result.pydantic

print(f"Primary issue: {report.primary_issue}")
print(f"Root cause: {report.root_cause}")
print(f"\nErrors found: {len(report.errors)}")
for error in report.errors:
    print(f"  [{error.severity}] {error.timestamp} — {error.message}")
print(f"\nAffected components: {report.affected_components}")

Primary issue: Deployment failure due to image pull error and exceeded deployment progress deadline.
Root cause: Image pull access was denied because the repository 'myapp:v1.2.3' does not exist or requires docker login.

Errors found: 5
  [ERROR] 2024-01-15 14:32:17.890 — Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
  [ERROR] 2024-01-15 14:32:18.123 — Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
  [ERROR] 2024-01-15 14:32:18.456 — Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
  [ERROR] 2024-01-15 14:32:25.901 — Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
  [CRITICAL] 2024-01-15 14:32:29.456 — Production deployment failed - rollback initiated

Affected components: ['myapp-deployment', 'myapp-deployment-7b8c9d5f4-abc12', 'myapp-service']


In [9]:
# Bonus: get the full output as JSON
print(report.model_dump_json(indent=2))

{
  "primary_issue": "Deployment failure due to image pull error and exceeded deployment progress deadline.",
  "root_cause": "Image pull access was denied because the repository 'myapp:v1.2.3' does not exist or requires docker login.",
  "errors": [
    {
      "message": "Pod myapp-deployment-7b8c9d5f4-abc12 failed to start",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:17.890"
    },
    {
      "message": "Failed to pull image \"myapp:v1.2.3\": pull access denied, repository does not exist or may require 'docker login'",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:18.123"
    },
    {
      "message": "Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:18.456"
    },
    {
      "message": "Deployment rollout failed: deployment \"myapp-deployment\" exceeded its progress deadline",
      "severity": "ERROR",
      "timestamp": "2024-01-15 14:32:25.901"
    },
    {
   